In [ ]:

# CA-FTX+: A Gated Feature-Tokenizer Transformer with Particle-Swarm-Optimized Hyperparameters for Employee Attrition Prediction
# Authors: Hedieh Sajedi , Abolfazl Khojasteh Abkenar

import os, sys, math, random, warnings, zipfile, copy, json, time
from itertools import combinations
warnings.filterwarnings("ignore")

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import RepeatedStratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    balanced_accuracy_score, matthews_corrcoef, roc_auc_score,
    average_precision_score, confusion_matrix, roc_curve, precision_recall_curve
)
from sklearn.manifold import TSNE

from scipy.stats import wilcoxon, friedmanchisquare, rankdata, spearmanr, pearsonr


def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except TypeError:
        pass


seed_everything(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)


CSV_PATH = "WA_Fn-UseC_-HR-Employee-Attrition.csv"   
N_FOLDS = 5
N_REPEATS = 5                 
EPOCHS = 200
BATCH_SIZE = 64
PATIENCE = 20
FOCAL_GAMMA_DEFAULT = 2.0
LABEL_SMOOTH = 0.03
N_ENSEMBLE = 3                 


PSO_PARTICLES = 8
PSO_ITERS = 6
PSO_EPOCHS = 60                
PSO_PATIENCE = 10
PSO_W, PSO_C1, PSO_C2 = 0.6, 1.4, 1.4   


OUT_DIR = "outputs"
FIG_DIR = os.path.join(OUT_DIR, "figures")
TAB_DIR = os.path.join(OUT_DIR, "tables")
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TAB_DIR, exist_ok=True)



JOURNAL_FONT = ["Times New Roman", "DejaVu Serif", "Georgia", "serif"]

mpl.rcParams.update({
    "font.family": "serif", "font.serif": JOURNAL_FONT,
    "axes.titlesize": 15, "axes.titleweight": "bold",
    "axes.labelsize": 12, "axes.edgecolor": "#3B3B3B", "axes.linewidth": 1.0,
    "xtick.labelsize": 10, "ytick.labelsize": 10,
    "legend.frameon": True, "legend.framealpha": 0.9,
    "figure.dpi": 150, "savefig.dpi": 300, "savefig.bbox": "tight",
    "axes.grid": True, "grid.alpha": 0.25, "grid.linestyle": "--",
})
sns.set_style("whitegrid", {"font.family": "serif", "font.serif": JOURNAL_FONT})



PALETTE = [
    "#1B4F72",  
    "#B03A2E",  
    "#1E8449",  
    "#B7950B",  
    "#6C3483",  
    "#117A65",  
    "#CA6F1E",  
    "#566573",  
]
sns.set_palette(PALETTE)


LW_PROPOSED = 3.0
LW_BASELINE = 1.5


def save_fig(fig, name):

    fig.savefig(os.path.join(FIG_DIR, f"{name}.png"), dpi=300, bbox_inches="tight")
    fig.savefig(os.path.join(FIG_DIR, f"{name}.pdf"), bbox_inches="tight")
    plt.close(fig)


def _df_to_latex_booktabs(df, name, float_format="%.4f", caption=None):

    def fmt_val(v):
        if isinstance(v, (int, np.integer)):
            return str(v)
        if isinstance(v, (float, np.floating)):
            if pd.isna(v):
                return ""
            return float_format % v
        return str(v)

    is_multi_cols = isinstance(df.columns, pd.MultiIndex)
    if is_multi_cols:
        col_labels = [" / ".join(str(x) for x in c) for c in df.columns]
    else:
        col_labels = [str(c) for c in df.columns]

    index_name = df.index.name or ""
    header = " & ".join([index_name] + col_labels) + r" \\"
    col_format = "l" + "r" * len(col_labels)

    lines = [
        r"\begin{table}[ht]", r"\centering",
        r"\begin{tabular}{%s}" % col_format,
        r"\toprule",
        header,
        r"\midrule",
    ]
    for idx, row in df.iterrows():
        idx_str = " / ".join(str(x) for x in idx) if isinstance(idx, tuple) else str(idx)
        cells = [fmt_val(v) for v in row.tolist()]
        lines.append(" & ".join([idx_str] + cells) + r" \\")
    lines += [r"\bottomrule", r"\end{tabular}"]
    if caption:
        lines.append(r"\caption{%s}" % caption)
    lines.append(r"\end{table}")

    with open(os.path.join(TAB_DIR, f"{name}.tex"), "w") as f:
        f.write("\n".join(lines))


def save_table(df, name, caption=None):
    df.to_csv(os.path.join(TAB_DIR, f"{name}.csv"), float_format="%.4f")

    try:
        numeric_df = df.select_dtypes(include=[np.number])
        if numeric_df.shape[1] == 0:
            raise ValueError("no numeric columns to style")
        styler = df.style.background_gradient(
            cmap="Blues", subset=numeric_df.columns, axis=0
        ).format(precision=4)
        styler.to_excel(os.path.join(TAB_DIR, f"{name}.xlsx"), engine="openpyxl")
    except Exception:
        try:
            df.to_excel(os.path.join(TAB_DIR, f"{name}.xlsx"), engine="openpyxl")
        except Exception:
            pass

    try:
        _df_to_latex_booktabs(df, name, float_format="%.4f", caption=caption)
    except Exception:
        pass



def load_ibm_hr_dataset(csv_path=CSV_PATH):
    if not os.path.isfile(csv_path):
        raise FileNotFoundError(
            f"Could not find '{csv_path}'. Download the dataset from "
            "https://www.kaggle.com/datasets/pavansubhasht/ibm-hr-analytics-attrition-dataset "
            "and place the CSV next to this script, or set CSV_PATH."
        )
    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]
    if df["Attrition"].dtype == object:
        df["Attrition"] = df["Attrition"].astype(str).str.strip().str.lower().map(
            {"yes": 1, "no": 0}
        )
    df["Attrition"] = df["Attrition"].astype(int)
    n_dupes = df.drop(columns=["EmployeeNumber"], errors="ignore").duplicated().sum()
    print(f"Loaded {csv_path} -- shape={df.shape}, "
          f"positive rate={df['Attrition'].mean():.3f}, exact duplicate rows={n_dupes}")
    if n_dupes > 0:
        print("WARNING: duplicate rows found. If any duplicate pair is split across "
              "train/test, that IS leakage. Consider df.drop_duplicates() before use.")
    return df, "Attrition"


CATEGORICAL_COLS = ["BusinessTravel", "Department", "EducationField",
                     "Gender", "JobRole", "MaritalStatus", "OverTime"]
DROP_COLS = ["EmployeeCount", "EmployeeNumber", "Over18", "StandardHours"]


def preprocess(df, target_col):
    df = df.copy().dropna(subset=[target_col])
    y = df[target_col].values.astype(np.int64)
    X = df.drop(columns=[target_col])
    X = X.drop(columns=[c for c in DROP_COLS if c in X.columns])
    X = X.drop(columns=[c for c in X.columns if X[c].nunique() <= 1])

    cat_cols = [c for c in CATEGORICAL_COLS if c in X.columns]
    num_cols = [c for c in X.columns if c not in cat_cols]

    for c in num_cols:
        X[c] = pd.to_numeric(X[c], errors="coerce")
        X[c] = X[c].fillna(X[c].median())
    for c in cat_cols:
        X[c] = X[c].astype(str).fillna("missing")

    feature_names = num_cols + cat_cols
    return X[num_cols].values.astype(np.float32), X[cat_cols].values, \
        y, feature_names, num_cols, cat_cols


def sanity_check_no_leakage(tr_idx, va_idx, test_idx):

    s_tr, s_va, s_te = set(tr_idx.tolist()), set(va_idx.tolist()), set(test_idx.tolist())
    assert s_tr.isdisjoint(s_te), "LEAKAGE: train/test index overlap."
    assert s_va.isdisjoint(s_te), "LEAKAGE: val/test index overlap."
    assert s_tr.isdisjoint(s_va), "LEAKAGE: train/val index overlap."



class FeatureTokenizer(nn.Module):
    def __init__(self, n_num, cat_cardinalities, d_token):
        super().__init__()
        self.n_num = n_num
        self.d_token = d_token
        if n_num > 0:
            self.num_weight = nn.Parameter(torch.empty(n_num, d_token))
            self.num_bias = nn.Parameter(torch.empty(n_num, d_token))
            nn.init.uniform_(self.num_weight, -1 / math.sqrt(d_token), 1 / math.sqrt(d_token))
            nn.init.uniform_(self.num_bias, -1 / math.sqrt(d_token), 1 / math.sqrt(d_token))
        self.cat_embeddings = nn.ModuleList(
            [nn.Embedding(card, d_token) for card in cat_cardinalities]
        )
        for emb in self.cat_embeddings:
            nn.init.normal_(emb.weight, std=0.02)
        self.cls = nn.Parameter(torch.zeros(1, 1, d_token))
        nn.init.normal_(self.cls, std=0.02)

    def forward(self, x_num, x_cat):
        tokens = []
        b = x_num.shape[0] if self.n_num > 0 else x_cat.shape[0]
        if self.n_num > 0:
            num_tok = x_num.unsqueeze(-1) * self.num_weight + self.num_bias
            tokens.append(num_tok)
        for i, emb in enumerate(self.cat_embeddings):
            tokens.append(emb(x_cat[:, i]).unsqueeze(1))
        cls = self.cls.expand(b, -1, -1)
        tokens = [cls] + tokens
        return torch.cat(tokens, dim=1)  
        


class GatedFeatureInteraction(nn.Module):
    def __init__(self, d_token, dropout):
        super().__init__()
        self.summary = nn.Linear(d_token, d_token)
        self.gate = nn.Sequential(
            nn.Linear(d_token * 2, d_token), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_token, d_token), nn.Sigmoid(),
        )

    def forward(self, tok):
        ctx = self.summary(tok.mean(dim=1, keepdim=True)).expand_as(tok)
        g = self.gate(torch.cat([tok, ctx], dim=-1))
        return tok * g


class TransformerEncoderBlock(nn.Module):
    def __init__(self, d_token, n_heads, ffn_mult, dropout, attn_dropout):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_token)
        self.attn = nn.MultiheadAttention(d_token, n_heads, dropout=attn_dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(d_token)
        self.ffn = nn.Sequential(
            nn.Linear(d_token, d_token * ffn_mult),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_token * ffn_mult, d_token),
        )
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        h = self.norm1(x)
        a, attn_w = self.attn(h, h, h, need_weights=True, average_attn_weights=True)
        x = x + self.drop(a)
        h = self.norm2(x)
        x = x + self.drop(self.ffn(h))
        return x, attn_w


class CA_FTX_Plus(nn.Module):
    def __init__(self, n_num, cat_cardinalities, d_token=32, n_heads=4,
                 n_layers=2, ffn_mult=2, dropout=0.25, attn_dropout=0.15,
                 use_gate=True):
        super().__init__()
        self.use_gate = use_gate
        self.tokenizer = FeatureTokenizer(n_num, cat_cardinalities, d_token)
        self.gate = GatedFeatureInteraction(d_token, dropout) if use_gate else None
        self.token_dropout = nn.Dropout(dropout * 0.5)
        self.blocks = nn.ModuleList([
            TransformerEncoderBlock(d_token, n_heads, ffn_mult, dropout, attn_dropout)
            for _ in range(n_layers)
        ])
        self.final_norm = nn.LayerNorm(d_token)
        self.head = nn.Sequential(
            nn.Linear(d_token, d_token), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_token, 1),
        )

    def forward(self, x_num, x_cat, return_attn=False):
        tok = self.tokenizer(x_num, x_cat)
        if self.gate is not None:
            tok = self.gate(tok)
        tok = self.token_dropout(tok)
        attn_maps = []
        for blk in self.blocks:
            tok, aw = blk(tok)
            attn_maps.append(aw)
        tok = self.final_norm(tok)
        cls_out = tok[:, 0]
        logit = self.head(cls_out).squeeze(-1)
        if return_attn:
            return logit, cls_out, attn_maps
        return logit, cls_out

    @torch.no_grad()
    def get_gate_values(self, x_num, x_cat):

        if self.gate is None:
            raise RuntimeError("get_gate_values() requires a model built with use_gate=True.")
        self.eval()
        tok = self.tokenizer(x_num, x_cat)
        ctx = self.gate.summary(tok.mean(dim=1, keepdim=True)).expand_as(tok)
        g = self.gate.gate(torch.cat([tok, ctx], dim=-1))
        g_feat = g[:, 1:, :].mean(dim=-1) 
        return g_feat.cpu().numpy()



class MLPBaseline(nn.Module):
    def __init__(self, n_num, cat_cardinalities, d_emb=8, hidden=128, dropout=0.25):
        super().__init__()
        self.cat_embeddings = nn.ModuleList([nn.Embedding(c, d_emb) for c in cat_cardinalities])
        in_dim = n_num + d_emb * len(cat_cardinalities)
        self.inp = nn.Sequential(nn.Linear(in_dim, hidden), nn.BatchNorm1d(hidden), nn.GELU(), nn.Dropout(dropout))
        self.block1 = nn.Sequential(nn.Linear(hidden, hidden), nn.BatchNorm1d(hidden), nn.GELU(), nn.Dropout(dropout))
        self.block2 = nn.Sequential(nn.Linear(hidden, hidden), nn.BatchNorm1d(hidden), nn.GELU(), nn.Dropout(dropout))
        self.out = nn.Linear(hidden, 1)

    def forward(self, x_num, x_cat, return_attn=False):
        embs = [emb(x_cat[:, i]) for i, emb in enumerate(self.cat_embeddings)]
        x = torch.cat([x_num] + embs, dim=1) if embs else x_num
        h = self.inp(x)
        h = h + self.block1(h)
        h = h + self.block2(h)
        logit = self.out(h).squeeze(-1)
        return logit, h


class PlainFeatureTransformer(nn.Module):
    def __init__(self, n_num, cat_cardinalities, d_token=32):
        super().__init__()
        self.tokenizer = FeatureTokenizer(n_num, cat_cardinalities, d_token)
        self.block = TransformerEncoderBlock(d_token, n_heads=2, ffn_mult=2, dropout=0.1, attn_dropout=0.1)
        self.head = nn.Linear(d_token, 1)

    def forward(self, x_num, x_cat, return_attn=False):
        tok = self.tokenizer(x_num, x_cat)
        tok, _ = self.block(tok)
        cls_out = tok[:, 0]
        return self.head(cls_out).squeeze(-1), cls_out


class WideShallowMLP(nn.Module):
    def __init__(self, n_num, cat_cardinalities, d_emb=8, hidden=256, dropout=0.3):
        super().__init__()
        self.cat_embeddings = nn.ModuleList([nn.Embedding(c, d_emb) for c in cat_cardinalities])
        in_dim = n_num + d_emb * len(cat_cardinalities)
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.BatchNorm1d(hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, 1),
        )

    def forward(self, x_num, x_cat, return_attn=False):
        embs = [emb(x_cat[:, i]) for i, emb in enumerate(self.cat_embeddings)]
        x = torch.cat([x_num] + embs, dim=1) if embs else x_num
        return self.net(x).squeeze(-1), x


BASELINE_REGISTRY = {
    "MLP (residual, regularized)": MLPBaseline,
    "Feature Transformer (plain, no reg.)": PlainFeatureTransformer,
    "Wide-Shallow MLP": WideShallowMLP,
}
PROPOSED_NAME = "CA-FTX+ (PSO-tuned, ensembled)"
ABLATION_NAME = "CA-FTX+ (no Gated Feature Interaction, ablation)"
COMPONENT_NAME = "Gated Feature Interaction"
BASELINE_NAMES = list(BASELINE_REGISTRY.keys())
MAIN_MODEL_NAMES = [PROPOSED_NAME] + BASELINE_NAMES
ALL_MODEL_NAMES = MAIN_MODEL_NAMES + [ABLATION_NAME]



class FocalLoss(nn.Module):
    def __init__(self, alpha, gamma=FOCAL_GAMMA_DEFAULT, label_smoothing=LABEL_SMOOTH):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.ls = label_smoothing

    def forward(self, logits, targets):
        targets = targets.float() * (1 - self.ls) + 0.5 * self.ls
        p = torch.sigmoid(logits)
        ce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        p_t = p * targets + (1 - p) * (1 - targets)
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        loss = alpha_t * (1 - p_t) ** self.gamma * ce
        return loss.mean()


class TabDataset(Dataset):
    def __init__(self, x_num, x_cat, y):
        self.x_num = torch.tensor(x_num, dtype=torch.float32)
        self.x_cat = torch.tensor(x_cat, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.x_num[idx], self.x_cat[idx], self.y[idx]


def mixup_batch(x_num, y, alpha):
    if alpha <= 0:
        return x_num, y, None, 1.0
    lam = np.random.beta(alpha, alpha)
    perm = torch.randperm(x_num.size(0), device=x_num.device)
    x_num_mixed = lam * x_num + (1 - lam) * x_num[perm]
    return x_num_mixed, y, perm, lam


def find_best_threshold(y_true, probs):
    best_t, best_f1 = 0.5, -1
    for t in np.linspace(0.05, 0.95, 181):
        f1 = f1_score(y_true, (probs >= t).astype(int), zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return best_t, best_f1



def train_one_model(model, train_loader, val_x, epochs, lr, weight_decay, patience,
                     alpha_focal, gamma_focal, mixup_alpha, use_mixup, is_ca_ftx):
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=max(10, epochs // 6))
    criterion = FocalLoss(alpha=alpha_focal, gamma=gamma_focal)

    val_xn, val_xc, val_y = val_x
    val_xn_t = torch.tensor(val_xn, dtype=torch.float32, device=DEVICE)
    val_xc_t = torch.tensor(val_xc, dtype=torch.long, device=DEVICE)

    best_auc, best_state, bad_epochs = -1, None, 0
    for epoch in range(epochs):
        model.train()
        for xn, xc, y in train_loader:
            xn, xc, y = xn.to(DEVICE), xc.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            if use_mixup and mixup_alpha > 0 and is_ca_ftx:
                xn_m, y_a, perm, lam = mixup_batch(xn, y, mixup_alpha)
                logit, _ = model(xn_m, xc)
                y_b = y[perm] if perm is not None else y
                loss = lam * criterion(logit, y_a) + (1 - lam) * criterion(logit, y_b)
            else:
                logit, _ = model(xn, xc)
                loss = criterion(logit, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            opt.step()
        sched.step()

        model.eval()
        with torch.no_grad():
            val_logit, _ = model(val_xn_t, val_xc_t)
            val_prob = torch.sigmoid(val_logit).cpu().numpy()
        try:
            val_auc = roc_auc_score(val_y, val_prob)
        except ValueError:
            val_auc = 0.5

        if val_auc > best_auc:
            best_auc = val_auc
            best_state = copy.deepcopy(model.state_dict())
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                break

    model.load_state_dict(best_state)
    return model, best_auc


def predict_prob(model, x_num, x_cat):
    model.eval()
    with torch.no_grad():
        xn_t = torch.tensor(x_num, dtype=torch.float32, device=DEVICE)
        xc_t = torch.tensor(x_cat, dtype=torch.long, device=DEVICE)
        logit, _ = model(xn_t, xc_t)
        return torch.sigmoid(logit).cpu().numpy()


def evaluate_from_prob(y, prob, threshold):
    pred = (prob >= threshold).astype(int)
    metrics = {
        "AUC_ROC": roc_auc_score(y, prob) if len(np.unique(y)) > 1 else float("nan"),
        "AUC_PR": average_precision_score(y, prob),
        "F1": f1_score(y, pred, zero_division=0),
        "Precision": precision_score(y, pred, zero_division=0),
        "Recall": recall_score(y, pred, zero_division=0),
        "BalancedAcc": balanced_accuracy_score(y, pred),
        "MCC": matthews_corrcoef(y, pred) if len(np.unique(pred)) > 1 else 0.0,
        "Accuracy": accuracy_score(y, pred),
        "Threshold": threshold,
    }
    return metrics, pred




ARCH_CHOICES = [  
    (16, 2, 1), (16, 4, 1), (24, 4, 1), (32, 4, 1), (32, 4, 2),
    (32, 8, 2), (48, 4, 2), (48, 8, 2), (64, 8, 2), (64, 8, 3),
]

PSO_BOUNDS = {
    "arch_idx":     (0, len(ARCH_CHOICES) - 1e-6),
    "dropout":      (0.10, 0.50),
    "attn_dropout": (0.05, 0.30),
    "log_lr":       (math.log10(1e-4), math.log10(3e-3)),
    "log_wd":       (math.log10(1e-4), math.log10(2e-2)),
    "focal_gamma":  (1.0, 3.0),
    "mixup_alpha":  (0.0, 0.5),
}


def decode_particle(pos):
    arch = ARCH_CHOICES[int(np.clip(round(pos["arch_idx"]), 0, len(ARCH_CHOICES) - 1))]
    return dict(
        d_token=arch[0], n_heads=arch[1], n_layers=arch[2],
        dropout=float(np.clip(pos["dropout"], *PSO_BOUNDS["dropout"])),
        attn_dropout=float(np.clip(pos["attn_dropout"], *PSO_BOUNDS["attn_dropout"])),
        lr=float(10 ** np.clip(pos["log_lr"], *PSO_BOUNDS["log_lr"])),
        weight_decay=float(10 ** np.clip(pos["log_wd"], *PSO_BOUNDS["log_wd"])),
        focal_gamma=float(np.clip(pos["focal_gamma"], *PSO_BOUNDS["focal_gamma"])),
        mixup_alpha=float(np.clip(pos["mixup_alpha"], *PSO_BOUNDS["mixup_alpha"])),
    )


def pso_fitness(cfg, n_num, cat_cardinalities, tr_data, val_data, alpha_focal):
    xn_tr, xc_tr, y_tr = tr_data
    seed_everything(SEED)
    model = CA_FTX_Plus(n_num, cat_cardinalities, d_token=cfg["d_token"], n_heads=cfg["n_heads"],
                         n_layers=cfg["n_layers"], dropout=cfg["dropout"],
                         attn_dropout=cfg["attn_dropout"]).to(DEVICE)
    loader = DataLoader(TabDataset(xn_tr, xc_tr, y_tr), batch_size=BATCH_SIZE, shuffle=True)
    model, _ = train_one_model(model, loader, val_data, PSO_EPOCHS, cfg["lr"], cfg["weight_decay"],
                                PSO_PATIENCE, alpha_focal, cfg["focal_gamma"], cfg["mixup_alpha"],
                                use_mixup=True, is_ca_ftx=True)
    val_xn, val_xc, val_y = val_data
    prob = predict_prob(model, val_xn, val_xc)
    thr, f1 = find_best_threshold(val_y, prob)
    return f1


def run_pso_search(n_num, cat_cardinalities, tr_data, val_data, alpha_focal,
                    n_particles=PSO_PARTICLES, n_iters=PSO_ITERS):
    print(f"\n--- PSO metaheuristic search: {n_particles} particles x {n_iters} iterations "
          f"({n_particles * n_iters} short trainings on an inner split) ---")
    keys = list(PSO_BOUNDS.keys())
    swarm = []
    for _ in range(n_particles):
        pos = {k: np.random.uniform(*PSO_BOUNDS[k]) for k in keys}
        vel = {k: 0.0 for k in keys}
        swarm.append({"pos": pos, "vel": vel, "best_pos": dict(pos), "best_score": -np.inf})

    global_best_pos, global_best_score = None, -np.inf
    history = []

    for it in range(n_iters):
        for p_i, particle in enumerate(swarm):
            cfg = decode_particle(particle["pos"])
            score = pso_fitness(cfg, n_num, cat_cardinalities, tr_data, val_data, alpha_focal)

            if score > particle["best_score"]:
                particle["best_score"] = score
                particle["best_pos"] = dict(particle["pos"])
            if score > global_best_score:
                global_best_score = score
                global_best_pos = dict(particle["pos"])

            print(f"  iter {it+1}/{n_iters} particle {p_i+1}/{n_particles}: "
                  f"val F1={score:.4f} arch={ (cfg['d_token'], cfg['n_heads'], cfg['n_layers']) } "
                  f"lr={cfg['lr']:.2e}")

        for particle in swarm:
            for k in keys:
                r1, r2 = np.random.rand(), np.random.rand()
                particle["vel"][k] = (
                    PSO_W * particle["vel"][k]
                    + PSO_C1 * r1 * (particle["best_pos"][k] - particle["pos"][k])
                    + PSO_C2 * r2 * (global_best_pos[k] - particle["pos"][k])
                )
                particle["pos"][k] = particle["pos"][k] + particle["vel"][k]
                lo, hi = PSO_BOUNDS[k]
                particle["pos"][k] = float(np.clip(particle["pos"][k], lo, hi))

        history.append({"iteration": it + 1, "best_f1_so_far": global_best_score})

    best_cfg = decode_particle(global_best_pos)
    print(f"--- PSO search done. Best inner-validation F1={global_best_score:.4f}, config={best_cfg} ---\n")
    return best_cfg, global_best_score, pd.DataFrame(history)







def run_experiment(df, target_col):
    x_num_raw, x_cat_raw, y, feature_names, num_cols, cat_cols = preprocess(df, target_col)

    ordinal = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    x_cat_full = ordinal.fit_transform(x_cat_raw).astype(np.int64)
    cat_cardinalities = [len(cats) for cats in ordinal.categories_]
    n_num = x_num_raw.shape[1]
    search_idx, _holdout_idx = train_test_split(
        np.arange(len(y)), test_size=0.20, stratify=y, random_state=SEED
    )
    s_tr_idx, s_val_idx = train_test_split(
        search_idx, test_size=0.20, stratify=y[search_idx], random_state=SEED
    )
    sanity_check_no_leakage(s_tr_idx, s_val_idx, _holdout_idx)

    s_scaler = StandardScaler()
    s_xn_tr = s_scaler.fit_transform(x_num_raw[s_tr_idx])
    s_xn_val = s_scaler.transform(x_num_raw[s_val_idx])
    s_xc_tr, s_xc_val = x_cat_full[s_tr_idx], x_cat_full[s_val_idx]
    s_y_tr, s_y_val = y[s_tr_idx], y[s_val_idx]
    s_alpha = 1 - s_y_tr.mean()

    best_cfg, best_search_f1, pso_history = run_pso_search(
        n_num, cat_cardinalities,
        (s_xn_tr, s_xc_tr, s_y_tr), (s_xn_val, s_xc_val, s_y_val), s_alpha,
    )
    save_table(pso_history.set_index("iteration"), "table05_pso_convergence")
    with open(os.path.join(TAB_DIR, "pso_best_config.json"), "w") as f:
        json.dump(best_cfg, f, indent=2)
    rskf = RepeatedStratifiedKFold(n_splits=N_FOLDS, n_repeats=N_REPEATS, random_state=SEED)

    all_names = list(ALL_MODEL_NAMES)
    fold_metrics = {name: [] for name in all_names}
    last_fold_curves = {}
    last_fold_artifacts = None

    for fold_i, (train_idx, test_idx) in enumerate(rskf.split(x_num_raw, y)):
        seed_everything(SEED + fold_i)

        tr_idx, va_idx = train_test_split(
            train_idx, test_size=0.15, stratify=y[train_idx], random_state=SEED + fold_i
        )
        sanity_check_no_leakage(tr_idx, va_idx, test_idx)

        scaler = StandardScaler()
        xn_tr = scaler.fit_transform(x_num_raw[tr_idx])
        xn_va = scaler.transform(x_num_raw[va_idx])
        xn_te = scaler.transform(x_num_raw[test_idx])

        xc_tr, xc_va, xc_te = x_cat_full[tr_idx], x_cat_full[va_idx], x_cat_full[test_idx]
        y_tr, y_va, y_te = y[tr_idx], y[va_idx], y[test_idx]

        pos_rate = y_tr.mean()
        alpha_focal = 1 - pos_rate
        train_loader = DataLoader(TabDataset(xn_tr, xc_tr, y_tr), batch_size=BATCH_SIZE,
                                   shuffle=True, drop_last=False)

        member_probs_val, member_probs_te = [], []
        ens_model_for_importance = None
        for m in range(N_ENSEMBLE):
            seed_everything(SEED + fold_i * 100 + m)
            model = CA_FTX_Plus(n_num, cat_cardinalities, d_token=best_cfg["d_token"],
                                 n_heads=best_cfg["n_heads"], n_layers=best_cfg["n_layers"],
                                 dropout=best_cfg["dropout"], attn_dropout=best_cfg["attn_dropout"]).to(DEVICE)
            model, _ = train_one_model(model, train_loader, (xn_va, xc_va, y_va), EPOCHS,
                                        best_cfg["lr"], best_cfg["weight_decay"], PATIENCE,
                                        alpha_focal, best_cfg["focal_gamma"], best_cfg["mixup_alpha"],
                                        use_mixup=True, is_ca_ftx=True)
            member_probs_val.append(predict_prob(model, xn_va, xc_va))
            member_probs_te.append(predict_prob(model, xn_te, xc_te))
            ens_model_for_importance = model  

        val_prob = np.mean(member_probs_val, axis=0)
        te_prob = np.mean(member_probs_te, axis=0)
        thr, _ = find_best_threshold(y_va, val_prob)  
        metrics, pred = evaluate_from_prob(y_te, te_prob, thr)
        fold_metrics[PROPOSED_NAME].append(metrics)

        if fold_i == rskf.get_n_splits() - 1:
            fpr, tpr, _ = roc_curve(y_te, te_prob)
            prec, rec, _ = precision_recall_curve(y_te, te_prob)
            last_fold_curves[PROPOSED_NAME] = dict(fpr=fpr, tpr=tpr, prec=prec, rec=rec,
                                                     prob=te_prob, pred=pred, y=y_te)
            last_fold_artifacts = dict(model=ens_model_for_importance, xn=xn_te, xc=xc_te,
                                        y=y_te, threshold=thr, scaler=scaler)


        seed_everything(SEED + fold_i * 100 + 999)
        ablation_model = CA_FTX_Plus(n_num, cat_cardinalities, d_token=best_cfg["d_token"],
                                      n_heads=best_cfg["n_heads"], n_layers=best_cfg["n_layers"],
                                      dropout=best_cfg["dropout"], attn_dropout=best_cfg["attn_dropout"],
                                      use_gate=False).to(DEVICE)
        ablation_model, _ = train_one_model(ablation_model, train_loader, (xn_va, xc_va, y_va), EPOCHS,
                                             best_cfg["lr"], best_cfg["weight_decay"], PATIENCE,
                                             alpha_focal, best_cfg["focal_gamma"], best_cfg["mixup_alpha"],
                                             use_mixup=True, is_ca_ftx=True)
        abl_val_prob = predict_prob(ablation_model, xn_va, xc_va)
        abl_thr, _ = find_best_threshold(y_va, abl_val_prob)
        abl_te_prob = predict_prob(ablation_model, xn_te, xc_te)
        abl_metrics, abl_pred = evaluate_from_prob(y_te, abl_te_prob, abl_thr)
        fold_metrics[ABLATION_NAME].append(abl_metrics)

        if fold_i == rskf.get_n_splits() - 1:
            fpr, tpr, _ = roc_curve(y_te, abl_te_prob)
            prec, rec, _ = precision_recall_curve(y_te, abl_te_prob)
            last_fold_curves[ABLATION_NAME] = dict(fpr=fpr, tpr=tpr, prec=prec, rec=rec,
                                                     prob=abl_te_prob, pred=abl_pred, y=y_te)

        for name, model_cls in BASELINE_REGISTRY.items():
            seed_everything(SEED + fold_i)
            model = model_cls(n_num, cat_cardinalities).to(DEVICE)
            model, _ = train_one_model(model, train_loader, (xn_va, xc_va, y_va), EPOCHS,
                                        3e-4, 5e-3, PATIENCE, alpha_focal, FOCAL_GAMMA_DEFAULT,
                                        0.0, use_mixup=False, is_ca_ftx=False)
            va_prob = predict_prob(model, xn_va, xc_va)
            thr_b, _ = find_best_threshold(y_va, va_prob)
            te_prob_b = predict_prob(model, xn_te, xc_te)
            metrics_b, pred_b = evaluate_from_prob(y_te, te_prob_b, thr_b)
            fold_metrics[name].append(metrics_b)

            if fold_i == rskf.get_n_splits() - 1:
                fpr, tpr, _ = roc_curve(y_te, te_prob_b)
                prec, rec, _ = precision_recall_curve(y_te, te_prob_b)
                last_fold_curves[name] = dict(fpr=fpr, tpr=tpr, prec=prec, rec=rec,
                                               prob=te_prob_b, pred=pred_b, y=y_te)

        print(f"Fold {fold_i + 1}/{rskf.get_n_splits()} done "
              f"({PROPOSED_NAME} F1={fold_metrics[PROPOSED_NAME][-1]['F1']:.4f}).")

    return fold_metrics, last_fold_curves, last_fold_artifacts, feature_names, \
        num_cols, cat_cols, cat_cardinalities, best_cfg




def aggregate_metrics(fold_metrics):
    rows_mean, rows_std = {}, {}
    for name, folds in fold_metrics.items():
        dfm = pd.DataFrame(folds)
        rows_mean[name] = dfm.mean()
        rows_std[name] = dfm.std()
    mean_df = pd.DataFrame(rows_mean).T
    std_df = pd.DataFrame(rows_std).T
    return mean_df, std_df


def plot_roc_pr(curves):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
    for i, (name, c) in enumerate(curves.items()):
        auc = roc_auc_score(c["y"], c["prob"])
        axes[0].plot(c["fpr"], c["tpr"], label=f"{name} (AUC={auc:.3f})", color=PALETTE[i % len(PALETTE)])
        ap = average_precision_score(c["y"], c["prob"])
        axes[1].plot(c["rec"], c["prec"], label=f"{name} (AP={ap:.3f})", color=PALETTE[i % len(PALETTE)])
    axes[0].plot([0, 1], [0, 1], "k--", alpha=0.4)
    axes[0].set_xlabel("False Positive Rate"); axes[0].set_ylabel("True Positive Rate")
    axes[0].set_title("ROC Curves (last fold)"); axes[0].legend(fontsize=8)
    axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
    axes[1].set_title("Precision-Recall Curves (last fold)"); axes[1].legend(fontsize=8)
    fig.tight_layout()
    save_fig(fig, "fig01_roc_pr_curves")


def plot_confusion(curves, ncols=3):

    names = list(curves.keys())
    n = len(names)
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 4.0 * nrows))
    axes = np.atleast_2d(axes).reshape(nrows, ncols)
    for i, name in enumerate(names):
        ax = axes[i // ncols, i % ncols]
        c = curves[name]
        cm = confusion_matrix(c["y"], c["pred"])
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax, cbar=False,
                    xticklabels=["Stay", "Leave"], yticklabels=["Stay", "Leave"])
        ax.set_title(name, fontsize=9)
        ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
    for i in range(n, nrows * ncols):
        axes[i // ncols, i % ncols].axis("off")
    fig.suptitle("Confusion Matrices by Model (last fold, threshold=0.5)", fontweight="bold")
    fig.tight_layout()
    save_fig(fig, "fig02_confusion_matrices")


def plot_cv_box(fold_metrics, metric="F1"):

    rows = []
    for name, folds in fold_metrics.items():
        for m in folds:
            rows.append({"Model": name, metric: m[metric]})
    d = pd.DataFrame(rows)
    model_order = list(fold_metrics.keys())
    palette_map = {name: (PALETTE[0] if name == PROPOSED_NAME else "#B5B5B5") for name in model_order}

    fig, ax = plt.subplots(figsize=(9, 5))
    sns.boxplot(data=d, x="Model", y=metric, ax=ax, order=model_order, palette=palette_map)
    sns.stripplot(data=d, x="Model", y=metric, ax=ax, order=model_order,
                  color="black", alpha=0.4, size=3)
    ax.set_title(f"Cross-Validated {metric} Distribution ({N_FOLDS}x{N_REPEATS}={N_FOLDS * N_REPEATS} folds)")
    plt.setp(ax.get_xticklabels(), rotation=35, ha="right", fontsize=8)
    fig.tight_layout()
    save_fig(fig, f"fig03_cv_boxplot_{metric}")


def plot_mean_f1_ranked(mean_df):
    fig, ax = plt.subplots(figsize=(8, 5))
    order = mean_df["F1"].sort_values(ascending=False).index
    ax.barh(order, mean_df.loc[order, "F1"], color=PALETTE[0])
    ax.set_xlabel("Mean F1 (threshold tuned per fold, on validation only)")
    ax.set_title(f"Mean F1 across {N_FOLDS}x{N_REPEATS} folds")
    fig.tight_layout()
    save_fig(fig, "fig04_mean_f1_ranked")


def plot_pso_convergence(pso_history):
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.plot(pso_history["iteration"], pso_history["best_f1_so_far"], marker="o", color=PALETTE[4])
    ax.set_xlabel("PSO iteration"); ax.set_ylabel("Best inner-validation F1 so far")
    ax.set_title("PSO Metaheuristic Search Convergence")
    fig.tight_layout()
    save_fig(fig, "fig06_pso_convergence")


def permutation_importance_proposed(artifacts, feature_names, num_cols, cat_cols):
    model, xn, xc, y = artifacts["model"], artifacts["xn"], artifacts["xc"], artifacts["y"]
    thr = artifacts["threshold"]
    base_prob = predict_prob(model, xn, xc)
    base_metrics, _ = evaluate_from_prob(y, base_prob, thr)
    base_auc = base_metrics["AUC_ROC"]

    importances = []
    rng = np.random.RandomState(SEED)
    for j, col in enumerate(num_cols):
        xn_perm = xn.copy()
        xn_perm[:, j] = rng.permutation(xn_perm[:, j])
        prob_p = predict_prob(model, xn_perm, xc)
        m, _ = evaluate_from_prob(y, prob_p, thr)
        importances.append((col, base_auc - m["AUC_ROC"]))
    for j, col in enumerate(cat_cols):
        xc_perm = xc.copy()
        xc_perm[:, j] = rng.permutation(xc_perm[:, j])
        prob_p = predict_prob(model, xn, xc_perm)
        m, _ = evaluate_from_prob(y, prob_p, thr)
        importances.append((col, base_auc - m["AUC_ROC"]))

    imp_df = pd.DataFrame(importances, columns=["Feature", "AUC_Drop"]).sort_values(
        "AUC_Drop", ascending=False).reset_index(drop=True)
    save_table(imp_df.set_index("Feature"), "table_permutation_importance")

    fig, ax = plt.subplots(figsize=(8, 9))
    top = imp_df.head(20).iloc[::-1]
    ax.barh(top["Feature"], top["AUC_Drop"], color=PALETTE[2])
    ax.set_xlabel("AUC drop when feature is permuted (higher = more important)")
    ax.set_title(f"Permutation Feature Importance -- {PROPOSED_NAME}")
    fig.tight_layout()
    save_fig(fig, "fig05_permutation_importance")
    return imp_df



def combined_mean_std_table(mean_df, std_df):
    combined = pd.DataFrame(index=mean_df.index, columns=mean_df.columns, dtype=object)
    for col in mean_df.columns:
        for idx in mean_df.index:
            combined.loc[idx, col] = f"{mean_df.loc[idx, col]:.4f} \u00b1 {std_df.loc[idx, col]:.4f}"
    return combined



def _paired_wilcoxon(a, b):

    diff = np.asarray(a) - np.asarray(b)
    if np.allclose(diff, 0):
        return np.nan, np.nan
    try:
        stat, p = wilcoxon(a, b)
    except ValueError:
        stat, p = np.nan, np.nan
    return stat, p


def pairwise_significance_table(fold_metrics, metric, proposed_name, baseline_names):

    prop_vals = np.array([m[metric] for m in fold_metrics[proposed_name]])
    rows = []
    for base_name in baseline_names:
        base_vals = np.array([m[metric] for m in fold_metrics[base_name]])
        stat, p = _paired_wilcoxon(prop_vals, base_vals)
        rows.append({
            "Comparison": f"{proposed_name} vs {base_name}",
            "Metric": metric,
            f"{proposed_name}_mean": prop_vals.mean(),
            f"{base_name}_mean": base_vals.mean(),
            "Wilcoxon_stat": stat,
            "p_value": p,
            "Significant (p<0.05)": bool(p < 0.05) if pd.notna(p) else False,
        })
    return pd.DataFrame(rows)



def relative_improvement_table(mean_df, proposed_name, baseline_names, eps=1e-9):
    prop_row = mean_df.loc[proposed_name]
    rel = pd.DataFrame(index=baseline_names, columns=mean_df.columns, dtype=float)
    for base_name in baseline_names:
        base_row = mean_df.loc[base_name]
        rel.loc[base_name] = (prop_row - base_row) / (base_row.abs() + eps) * 100.0
    return rel



def ablation_comparison_table(mean_df, proposed_name, ablation_name):

    return mean_df.loc[[proposed_name, ablation_name]]



def bootstrap_ci_table(fold_metrics, metrics, model_names, n_resamples=5000, ci=0.95, seed=SEED):

    rng = np.random.RandomState(seed)
    lo_pct, hi_pct = (1 - ci) / 2 * 100, (1 + ci) / 2 * 100
    cols = pd.MultiIndex.from_product([metrics, ["mean", "CI_low", "CI_high"]])
    out = pd.DataFrame(index=model_names, columns=cols, dtype=float)
    for name in model_names:
        for metric in metrics:
            vals = np.array([m[metric] for m in fold_metrics[name]])
            point_mean = vals.mean()
            boot_means = np.array([
                rng.choice(vals, size=len(vals), replace=True).mean()
                for _ in range(n_resamples)
            ])
            ci_low, ci_high = np.percentile(boot_means, [lo_pct, hi_pct])
            out.loc[name, (metric, "mean")] = point_mean
            out.loc[name, (metric, "CI_low")] = ci_low
            out.loc[name, (metric, "CI_high")] = ci_high
    return out



def cohend_effect_size_table(fold_metrics, metrics, proposed_name, baseline_names, eps=1e-9):

    rows = {}
    for base_name in baseline_names:
        row = {}
        for metric in metrics:
            prop_vals = np.array([m[metric] for m in fold_metrics[proposed_name]])
            base_vals = np.array([m[metric] for m in fold_metrics[base_name]])
            diff = prop_vals - base_vals
            row[metric] = diff.mean() / (diff.std(ddof=1) + eps)
        rows[f"{proposed_name} vs {base_name}"] = row
    return pd.DataFrame(rows).T



NEMENYI_Q_ALPHA_005 = {
    2: 1.960, 3: 2.343, 4: 2.569, 5: 2.728, 6: 2.850,
    7: 2.949, 8: 3.031, 9: 3.102, 10: 3.164,
}


def nemenyi_critical_difference(k, n_folds, alpha=0.05):
    q_alpha = NEMENYI_Q_ALPHA_005.get(k, NEMENYI_Q_ALPHA_005[10])
    return q_alpha * math.sqrt(k * (k + 1) / (6.0 * n_folds))


def friedman_nemenyi_tables(fold_metrics, metric, model_names):

    n_folds = len(fold_metrics[model_names[0]])
    k = len(model_names)
    values = np.array([[fold_metrics[name][f][metric] for name in model_names]
                        for f in range(n_folds)])
    ranks = np.array([rankdata(-values[f, :]) for f in range(n_folds)])
    avg_rank = pd.Series(ranks.mean(axis=0), index=model_names, name="Avg_Rank").sort_values()

    try:
        friedman_stat, friedman_p = friedmanchisquare(*[values[:, i] for i in range(k)])
    except ValueError:
        friedman_stat, friedman_p = np.nan, np.nan

    cd = nemenyi_critical_difference(k, n_folds)

    ranks_table = avg_rank.to_frame()
    ranks_table.attrs["Friedman_stat"] = friedman_stat
    ranks_table.attrs["Friedman_p"] = friedman_p
    ranks_table.attrs["n_models"] = k
    ranks_table.attrs["n_folds"] = n_folds
    ranks_table.attrs["CD"] = cd

    summary_table = pd.DataFrame([{
        "Metric": metric,
        "Friedman_stat": friedman_stat,
        "Friedman_p": friedman_p,
        "Significant (p<0.05)": bool(friedman_p < 0.05) if pd.notna(friedman_p) else False,
        "n_models": k,
        "n_folds": n_folds,
        "Nemenyi_CD": cd,
    }])
    return ranks_table, summary_table



def complexity_latency_table(n_num, cat_cardinalities, best_cfg, batch_size=64,
                              n_warmup=5, n_repeats=30):


    def build(name):
        if name == PROPOSED_NAME:
            return CA_FTX_Plus(n_num, cat_cardinalities, d_token=best_cfg["d_token"],
                                n_heads=best_cfg["n_heads"], n_layers=best_cfg["n_layers"],
                                dropout=best_cfg["dropout"], attn_dropout=best_cfg["attn_dropout"])
        if name == ABLATION_NAME:
            return CA_FTX_Plus(n_num, cat_cardinalities, d_token=best_cfg["d_token"],
                                n_heads=best_cfg["n_heads"], n_layers=best_cfg["n_layers"],
                                dropout=best_cfg["dropout"], attn_dropout=best_cfg["attn_dropout"],
                                use_gate=False)
        return BASELINE_REGISTRY[name](n_num, cat_cardinalities)

    rng = np.random.RandomState(SEED)
    x_num_dummy = rng.randn(batch_size, n_num).astype(np.float32)
    if cat_cardinalities:
        x_cat_dummy = np.stack(
            [rng.randint(0, c, size=batch_size) for c in cat_cardinalities], axis=1
        ).astype(np.int64)
    else:
        x_cat_dummy = np.zeros((batch_size, 0), dtype=np.int64)

    rows = []
    for name in ALL_MODEL_NAMES:
        seed_everything(SEED)
        model = build(name).to(DEVICE)
        model.eval()
        total_params = sum(p.numel() for p in model.parameters())
        trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

        xn_t = torch.tensor(x_num_dummy, dtype=torch.float32, device=DEVICE)
        xc_t = torch.tensor(x_cat_dummy, dtype=torch.long, device=DEVICE)
        with torch.no_grad():
            for _ in range(n_warmup):
                model(xn_t, xc_t)
            if DEVICE.type == "cuda":
                torch.cuda.synchronize()
            t0 = time.perf_counter()
            for _ in range(n_repeats):
                model(xn_t, xc_t)
            if DEVICE.type == "cuda":
                torch.cuda.synchronize()
            t1 = time.perf_counter()

        ms_per_batch = (t1 - t0) / n_repeats * 1000.0
        us_per_sample = ms_per_batch * 1000.0 / batch_size
        rows.append({
            "Model": name, "Total_Params": total_params, "Trainable_Params": trainable_params,
            "Inference_ms_per_batch": ms_per_batch, "Inference_us_per_sample": us_per_sample,
        })
    return pd.DataFrame(rows).set_index("Model")



def _sig_stars(p):
    if pd.isna(p):
        return ""
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return ""


def publication_summary_table(fold_metrics, mean_df, std_df, metrics, proposed_name, baseline_names):

    model_names = [proposed_name] + baseline_names
    plain = pd.DataFrame(index=model_names, columns=metrics, dtype=object)
    is_best = pd.DataFrame(index=model_names, columns=metrics, dtype=bool)

    for metric in metrics:
        best_model = mean_df.loc[model_names, metric].idxmax()
        prop_vals = np.array([m[metric] for m in fold_metrics[proposed_name]])
        for name in model_names:
            cell = f"{mean_df.loc[name, metric]:.4f} \u00b1 {std_df.loc[name, metric]:.4f}"
            if name != proposed_name:
                base_vals = np.array([m[metric] for m in fold_metrics[name]])
                try:
                    _, p = wilcoxon(prop_vals, base_vals)
                except ValueError:
                    p = np.nan
                cell += _sig_stars(p)
            plain.loc[name, metric] = cell
            is_best.loc[name, metric] = (name == best_model)

    col_format = "l" + "r" * len(metrics)
    lines = [r"\begin{table}[ht]", r"\centering", r"\begin{tabular}{%s}" % col_format,
             r"\toprule", " & ".join(["Model"] + metrics) + r" \\", r"\midrule"]
    for name in model_names:
        cells = []
        for metric in metrics:
            val = plain.loc[name, metric]
            cells.append((r"\textbf{%s}" % val) if is_best.loc[name, metric] else val)
        lines.append(" & ".join([name] + cells) + r" \\")
    lines += [r"\bottomrule", r"\end{tabular}",
              r"\caption{Cell format: mean $\pm$ std. Stars mark paired Wilcoxon "
              r"significance vs.\ the proposed model (*** p$<$0.001, ** p$<$0.01, "
              r"* p$<$0.05). The best value in each column is in \textbf{bold}.}",
              r"\end{table}"]
    latex_str = "\n".join(lines)
    with open(os.path.join(TAB_DIR, "table16_publication_summary.tex"), "w") as f:
        f.write(latex_str)

    return plain, latex_str



def _baseline_values(xn, xc):

    baseline_num = np.zeros(xn.shape[1], dtype=np.float32)
    if xc.shape[1] > 0:
        baseline_cat = np.array(
            [np.bincount(xc[:, j]).argmax() for j in range(xc.shape[1])], dtype=np.int64
        )
    else:
        baseline_cat = np.zeros(0, dtype=np.int64)
    return baseline_num, baseline_cat


def shap_style_attributions(model, xn, xc):

    baseline_num, baseline_cat = _baseline_values(xn, xc)
    base_prob = predict_prob(model, xn, xc)
    n_num, n_cat = xn.shape[1], xc.shape[1]
    attrs = np.zeros((xn.shape[0], n_num + n_cat), dtype=np.float32)
    for j in range(n_num):
        xn_occ = xn.copy(); xn_occ[:, j] = baseline_num[j]
        attrs[:, j] = base_prob - predict_prob(model, xn_occ, xc)
    for j in range(n_cat):
        xc_occ = xc.copy(); xc_occ[:, j] = baseline_cat[j]
        attrs[:, n_num + j] = base_prob - predict_prob(model, xn, xc_occ)
    return attrs


def integrated_gradients_attributions(model, xn, xc, n_steps=32):

    model.eval()
    n_num, n_cat = xn.shape[1], xc.shape[1]
    xn_t = torch.tensor(xn, dtype=torch.float32, device=DEVICE)
    xc_t = torch.tensor(xc, dtype=torch.long, device=DEVICE)
    baseline_t = torch.zeros_like(xn_t)

    ig_accum = torch.zeros_like(xn_t)
    for step in range(1, n_steps + 1):
        alpha = step / n_steps
        x_interp = (baseline_t + alpha * (xn_t - baseline_t)).clone().detach().requires_grad_(True)
        logit, _ = model(x_interp, xc_t)
        prob = torch.sigmoid(logit)
        grad = torch.autograd.grad(prob.sum(), x_interp)[0]
        ig_accum += grad
    ig_num = (ig_accum / n_steps * (xn_t - baseline_t)).detach().cpu().numpy()

    if n_cat > 0:
        _, baseline_cat = _baseline_values(xn, xc)
        base_prob = predict_prob(model, xn, xc)
        ig_cat = np.zeros((xn.shape[0], n_cat), dtype=np.float32)
        for j in range(n_cat):
            xc_occ = xc.copy(); xc_occ[:, j] = baseline_cat[j]
            ig_cat[:, j] = base_prob - predict_prob(model, xn, xc_occ)
        return np.concatenate([ig_num, ig_cat], axis=1)
    return ig_num


def extract_feature_salience(model, xn, xc, feature_names):
    xn_t = torch.tensor(xn, dtype=torch.float32, device=DEVICE)
    xc_t = torch.tensor(xc, dtype=torch.long, device=DEVICE)
    g = model.get_gate_values(xn_t, xc_t)  
    salience = pd.Series(g.mean(axis=0), index=feature_names, name="Salience")
    return salience.sort_values(ascending=False).to_frame()



def plot_feature_salience_bar(salience_table, top_k=20):
    top = salience_table["Salience"].head(top_k).iloc[::-1]
    fig, ax = plt.subplots(figsize=(8, 9))
    ax.barh(top.index, top.values, color=PALETTE[0], edgecolor="black")
    ax.set_xlabel("Learned gate value (sigmoid output)")
    ax.set_title(f"Top-{top_k} Learned Feature Salience -- {PROPOSED_NAME}")
    fig.tight_layout()
    save_fig(fig, "fig07_feature_salience_bar")



def plot_shap_global_bar(shap_attrs, feature_names, top_k=15):
    mean_abs = pd.Series(np.abs(shap_attrs).mean(axis=0), index=feature_names)
    top = mean_abs.sort_values(ascending=False).head(top_k).iloc[::-1]
    fig, ax = plt.subplots(figsize=(8, 7))
    ax.barh(top.index, top.values, color=PALETTE[0], edgecolor="black")
    ax.set_xlabel("Mean |Attribution|")
    ax.set_title(f"Global Feature Importance (SHAP-style) -- {PROPOSED_NAME}")
    fig.tight_layout()
    save_fig(fig, "fig08_shap_global_importance")
    return mean_abs



def plot_ig_directional_bar(ig_attrs, feature_names, top_k=15):
    mean_signed = pd.Series(ig_attrs.mean(axis=0), index=feature_names)
    mean_abs = mean_signed.abs()
    top_feats = mean_abs.sort_values(ascending=False).head(top_k).index
    top = mean_signed.loc[top_feats].sort_values().iloc[::-1]
    colors = [PALETTE[1] if v >= 0 else PALETTE[5] for v in top.values]
    fig, ax = plt.subplots(figsize=(8, 7))
    ax.barh(top.index, top.values, color=colors, edgecolor="black")
    ax.axvline(0, color="black", linewidth=1.0)
    ax.set_xlabel("Mean Signed Attribution (Integrated Gradients)")
    ax.set_title(f"Directional Attribution -- {PROPOSED_NAME}\n"
                 f"({PALETTE[1]}=pushes toward positive class, {PALETTE[5]}=pushes away)")
    fig.tight_layout()
    save_fig(fig, "fig09_ig_directional")
    return mean_signed



def plot_calibration_curves(curves, n_bins=10):
    fig, ax = plt.subplots(figsize=(7, 6))
    for name, c in curves.items():
        prob, y = c["prob"], c["y"]
        bin_edges = np.linspace(0, 1, n_bins + 1)
        bin_idx = np.digitize(prob, bin_edges[1:-1])
        obs_freq, mean_pred = [], []
        for b in range(n_bins):
            mask = bin_idx == b
            if mask.sum() == 0:
                continue
            obs_freq.append(y[mask].mean())
            mean_pred.append(prob[mask].mean())
        lw = LW_PROPOSED if name == PROPOSED_NAME else LW_BASELINE
        ax.plot(mean_pred, obs_freq, marker="o", linewidth=lw, label=name)
    ax.plot([0, 1], [0, 1], "k--", alpha=0.5, label="Perfect calibration")
    ax.set_xlabel("Mean Predicted Probability"); ax.set_ylabel("Observed Frequency")
    ax.set_title("Calibration / Reliability Curves (last fold)")
    ax.legend(fontsize=8)
    fig.tight_layout()
    save_fig(fig, "fig10_calibration")



def plot_dataset_overview(df, target_col, num_cols, negative_label="Stay", positive_label="Leave"):
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    counts = df[target_col].value_counts().sort_index()
    total = counts.sum()
    labels = [negative_label, positive_label]
    bars = axes[0].bar(labels, counts.values, color=[PALETTE[0], PALETTE[1]])
    for bar, cnt in zip(bars, counts.values):
        pct = cnt / total * 100
        axes[0].annotate(f"{cnt}\n({pct:.1f}%)", (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                          ha="center", va="bottom", fontsize=10)
    axes[0].set_title("Target Class Balance")
    axes[0].set_ylabel("Count")

    corr = df[num_cols].corr()
    sns.heatmap(corr, cmap="RdBu_r", center=0, annot=True, fmt=".2f",
                annot_kws={"size": 5}, ax=axes[1], cbar=True, square=True)
    axes[1].set_title("Feature Correlation")

    fig.suptitle("Dataset Overview: Class Balance & Feature Correlation", fontweight="bold")
    fig.tight_layout()
    save_fig(fig, "fig12_dataset_overview")



def plot_forest(bootstrap_df, metrics, proposed_name, ablation_name, baseline_names, nrows=2, ncols=3):
    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4.2 * nrows))
    axes = np.atleast_2d(axes).reshape(nrows, ncols)
    all_models = [proposed_name, ablation_name] + baseline_names

    for i, metric in enumerate(metrics):
        ax = axes[i // ncols, i % ncols]
        sub = bootstrap_df[metric].loc[all_models].sort_values("mean")
        colors = []
        for name in sub.index:
            if name == proposed_name:
                colors.append(PALETTE[0])
            elif name == ablation_name:
                colors.append(PALETTE[4])
            else:
                colors.append("#9E9E9E")
        y_pos = np.arange(len(sub))
        xerr = np.vstack([sub["mean"] - sub["CI_low"], sub["CI_high"] - sub["mean"]])
        ax.errorbar(sub["mean"], y_pos, xerr=xerr, fmt="o", ecolor="black",
                    capsize=3, markersize=0)
        for yi, (name, color) in enumerate(zip(sub.index, colors)):
            ax.scatter(sub.loc[name, "mean"], yi, color=color, zorder=3, s=60)
        ax.axvline(bootstrap_df[metric].loc[proposed_name, "mean"], color=PALETTE[0],
                   linestyle="--", alpha=0.6)
        ax.set_yticks(y_pos); ax.set_yticklabels(sub.index, fontsize=7)
        ax.set_title(metric, fontsize=11)

    for i in range(len(metrics), nrows * ncols):
        axes[i // ncols, i % ncols].axis("off")

    fig.suptitle("Model Comparison: Mean \u00b1 95% Bootstrap CI (Forest Plot)", fontweight="bold")
    fig.tight_layout()
    save_fig(fig, "fig13_forest_plot")



def plot_radar(mean_df, metrics, proposed_name):
    norm = (mean_df[metrics] - mean_df[metrics].min()) / (mean_df[metrics].max() - mean_df[metrics].min() + 1e-9)
    angles = np.linspace(0, 2 * math.pi, len(metrics), endpoint=False).tolist()
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
    for i, name in enumerate(norm.index):
        values = norm.loc[name].tolist()
        values += values[:1]
        is_proposed = (name == proposed_name)
        ax.plot(angles, values, linewidth=LW_PROPOSED if is_proposed else LW_BASELINE * 0.8,
                color=PALETTE[0] if is_proposed else PALETTE[(i + 1) % len(PALETTE)],
                label=name)
        ax.fill(angles, values, alpha=0.25 if is_proposed else 0.08,
                color=PALETTE[0] if is_proposed else PALETTE[(i + 1) % len(PALETTE)])
    ax.set_xticks(angles[:-1]); ax.set_xticklabels(metrics, fontsize=9)
    ax.set_yticklabels([])
    ax.set_title("Min-Max Normalized Performance Profile", y=1.08)
    ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.1), fontsize=8)
    fig.tight_layout()
    save_fig(fig, "fig14_radar_chart")



def plot_cd_diagram(ranks_table, metric_name):
    avg_rank = ranks_table["Avg_Rank"].sort_values()
    k = ranks_table.attrs["n_models"]
    cd = ranks_table.attrs["CD"]
    friedman_stat = ranks_table.attrs["Friedman_stat"]
    friedman_p = ranks_table.attrs["Friedman_p"]
    n_folds = ranks_table.attrs["n_folds"]

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot([1, k], [0, 0], color="black", linewidth=1.2)
    for r in range(1, k + 1):
        ax.plot([r, r], [-0.03, 0.03], color="black", linewidth=1.0)
        ax.text(r, 0.07, str(r), ha="center", fontsize=9)

    names = list(avg_rank.index)
    for i, (name, rank) in enumerate(avg_rank.items()):
        side = 1 if i % 2 == 0 else -1
        stem_len = 0.25 + 0.12 * (i // 2)
        y_stem = side * stem_len
        color = PALETTE[i % len(PALETTE)]
        ax.plot([rank, rank], [0, y_stem], color=color, linewidth=1.2)
        ax.text(rank, y_stem + (0.03 * side), f"{name} ({rank:.2f})",
                ha="center", va="bottom" if side > 0 else "top", fontsize=8, color=color)

    cd_y = -max(0.25 + 0.12 * (len(names) // 2), 0.5) - 0.15
    ax.plot([1, 1 + cd], [cd_y, cd_y], color="black", linewidth=1.5)
    ax.plot([1, 1], [cd_y - 0.02, cd_y + 0.02], color="black", linewidth=1.5)
    ax.plot([1 + cd, 1 + cd], [cd_y - 0.02, cd_y + 0.02], color="black", linewidth=1.5)
    ax.text(1 + cd / 2, cd_y - 0.05, f"CD = {cd:.3f}", ha="center", fontsize=9)

    sorted_ranks = avg_rank.sort_values()
    clique_y = max(0.25 + 0.12 * (len(names) // 2), 0.5) + 0.15
    i = 0
    vals = sorted_ranks.values
    idxs = sorted_ranks.index
    while i < len(vals):
        j = i
        while j + 1 < len(vals) and vals[j + 1] - vals[i] <= cd:
            j += 1
        if j > i:
            ax.plot([vals[i], vals[j]], [clique_y, clique_y], color="gray", linewidth=4, alpha=0.6)
            clique_y += 0.12
        i = j + 1

    ax.set_ylim(cd_y - 0.15, clique_y + 0.15)
    ax.axis("off")
    ax.set_title(f"Critical Difference Diagram -- {metric_name}\n"
                 f"Friedman \u03c7\u00b2={friedman_stat:.3f}, p={friedman_p:.4f}, "
                 f"n_folds={n_folds}, \u03b1=0.05")
    fig.tight_layout()
    save_fig(fig, f"fig15_cd_diagram_{metric_name}")



def plot_spaghetti(fold_metrics, metric, model_names):
    n_folds = len(fold_metrics[model_names[0]])
    values = np.array([[fold_metrics[name][f][metric] for name in model_names] for f in range(n_folds)])
    cmap = plt.get_cmap("viridis")
    fig, ax = plt.subplots(figsize=(10, 6))
    x = np.arange(len(model_names))
    for f in range(n_folds):
        ax.plot(x, values[f, :], color=cmap(f / max(n_folds - 1, 1)), alpha=0.5, linewidth=1.0)
    ax.plot(x, values.mean(axis=0), color="black", linewidth=3.0, label="Mean across folds")
    ax.set_xticks(x); ax.set_xticklabels(model_names, rotation=35, ha="right", fontsize=8)
    ax.set_ylabel(metric)
    ax.set_title(f"Per-Fold Consistency -- {metric} ({n_folds} folds)")
    ax.legend(fontsize=8)
    fig.tight_layout()
    save_fig(fig, f"fig16_spaghetti_{metric}")



def plot_ablation_slope(fold_metrics, metrics, proposed_name, ablation_name):
    n_metrics = len(metrics)
    fig, axes = plt.subplots(1, n_metrics, figsize=(4.5 * n_metrics, 5))
    axes = np.atleast_1d(axes)
    for ax, metric in zip(axes, metrics):
        without_vals = np.array([m[metric] for m in fold_metrics[ablation_name]])
        with_vals = np.array([m[metric] for m in fold_metrics[proposed_name]])
        for wo, wi in zip(without_vals, with_vals):
            color = PALETTE[1] if wi >= wo else PALETTE[5]
            ax.plot([0, 1], [wo, wi], color=color, alpha=0.4, linewidth=1.0)
        ax.plot([0, 1], [without_vals.mean(), with_vals.mean()], color="black", linewidth=3.0)
        try:
            _, p = wilcoxon(with_vals, without_vals)
        except ValueError:
            p = np.nan
        ax.set_xticks([0, 1]); ax.set_xticklabels([f"Without\n{COMPONENT_NAME}", "With\n" + COMPONENT_NAME])
        ax.set_ylabel(metric)
        p_str = f"{p:.4f}" if pd.notna(p) else "n/a"
        ax.set_title(f"{metric}\n(Wilcoxon p={p_str})", fontsize=10)
    fig.suptitle(f"Paired Per-Fold Ablation Effect: {COMPONENT_NAME}", fontweight="bold")
    fig.tight_layout()
    save_fig(fig, "fig17_ablation_slope")



def plot_tsne_embedding(model, xn, xc, y, prob):
    xn_t = torch.tensor(xn, dtype=torch.float32, device=DEVICE)
    xc_t = torch.tensor(xc, dtype=torch.long, device=DEVICE)
    model.eval()
    with torch.no_grad():
        _, cls_out = model(xn_t, xc_t)
    emb = cls_out.cpu().numpy()

    n = emb.shape[0]
    perplexity = min(30, max(5, n // 4))
    tsne = TSNE(n_components=2, perplexity=perplexity, init="pca", random_state=SEED)
    proj = tsne.fit_transform(emb)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
    for cls_val, label, color in [(0, "Stay", PALETTE[0]), (1, "Leave", PALETTE[1])]:
        mask = y == cls_val
        axes[0].scatter(proj[mask, 0], proj[mask, 1], color=color, label=label, alpha=0.7, s=20)
    axes[0].set_title("Colored by True Class"); axes[0].legend(fontsize=8)

    sc = axes[1].scatter(proj[:, 0], proj[:, 1], c=prob, cmap="coolwarm", alpha=0.8, s=20)
    fig.colorbar(sc, ax=axes[1], label="Predicted P(Leave)")
    axes[1].set_title("Colored by Predicted Probability")

    fig.suptitle(f"Learned Embedding t-SNE -- {PROPOSED_NAME} (qualitative class-separation check)",
                 fontweight="bold")
    fig.tight_layout()
    save_fig(fig, "fig18_tsne_embedding")




def plot_beeswarm(shap_attrs, xn, xc, feature_names, num_cols, top_k=15):
    n_num = len(num_cols)

    feat_vals = np.concatenate([xn, xc.astype(np.float32)], axis=1)
    mean_abs = np.abs(shap_attrs).mean(axis=0)
    top_idx = np.argsort(mean_abs)[::-1][:top_k]

    fig, ax = plt.subplots(figsize=(9, 8))
    for row, j in enumerate(top_idx[::-1]):
        vals = feat_vals[:, j]
        lo, hi = np.percentile(vals, [1, 99])
        clipped = np.clip(vals, lo, hi)
        scaled = (clipped - clipped.min()) / (clipped.max() - clipped.min() + 1e-9)
        jitter = (np.random.RandomState(SEED).rand(len(vals)) - 0.5) * 0.6
        sc = ax.scatter(shap_attrs[:, j], row + jitter, c=scaled, cmap="coolwarm", s=10, alpha=0.7)
    ax.set_yticks(range(len(top_idx)))
    ax.set_yticklabels([feature_names[j] for j in top_idx[::-1]], fontsize=8)
    ax.axvline(0, color="black", linewidth=1.0)
    fig.colorbar(sc, ax=ax, label="Feature value (low -> high)")
    ax.set_xlabel("SHAP-style Attribution")
    ax.set_title(f"SHAP-Style Beeswarm -- Top-{top_k} Features -- {PROPOSED_NAME}")
    fig.tight_layout()
    save_fig(fig, "fig19_beeswarm")



def plot_explanation_agreement(shap_attrs, ig_attrs, feature_names, top_k=15):
    shap_abs = pd.Series(np.abs(shap_attrs).mean(axis=0), index=feature_names)
    ig_abs = pd.Series(np.abs(ig_attrs).mean(axis=0), index=feature_names)
    combined_rank = (shap_abs + ig_abs).sort_values(ascending=False)
    top_feats = combined_rank.head(top_k).index

    x = shap_abs.loc[top_feats].values
    yv = ig_abs.loc[top_feats].values
    rho, _ = spearmanr(shap_abs.values, ig_abs.values)
    r, _ = pearsonr(shap_abs.values, ig_abs.values)

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.scatter(x, yv, color=PALETTE[0], s=40)
    for feat, xi, yi in zip(top_feats, x, yv):
        ax.annotate(feat, (xi, yi), fontsize=7, xytext=(3, 3), textcoords="offset points")
    lims = [0, max(x.max(), yv.max()) * 1.1]
    ax.plot(lims, lims, "k--", alpha=0.5, label="Perfect agreement")
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_xlabel("Mean |Attribution| -- SHAP-style"); ax.set_ylabel("Mean |Attribution| -- Integrated Gradients")
    ax.set_title(f"Cross-Method Explanation Agreement\nSpearman \u03c1={rho:.3f}, Pearson r={r:.3f}")
    ax.legend(fontsize=8)
    fig.tight_layout()
    save_fig(fig, "fig20_explanation_agreement")



def plot_local_waterfall(y, prob, pred, attrs, feature_names, base_rate, top_k=10,
                          negative_label="Stay", positive_label="Leave"):
    correct = pred == y
    pos_mask = correct & (y == 1)
    neg_mask = correct & (y == 0)

    cases = []
    if pos_mask.any():
        idx = np.arange(len(y))[pos_mask][np.argmax(prob[pos_mask])]
        cases.append((idx, f"Correctly predicted: {positive_label}"))
    if neg_mask.any():
        idx = np.arange(len(y))[neg_mask][np.argmin(prob[neg_mask])]
        cases.append((idx, f"Correctly predicted: {negative_label}"))

    if not cases:
        return

    fig, axes = plt.subplots(1, len(cases), figsize=(8 * len(cases), 6))
    axes = np.atleast_1d(axes)
    for ax, (idx, title) in zip(axes, cases):
        row = pd.Series(attrs[idx], index=feature_names)
        top = row.reindex(row.abs().sort_values(ascending=False).head(top_k).index)
        top = top.iloc[::-1]
        cumulative = base_rate
        for i, (feat, val) in enumerate(top.items()):
            color = PALETTE[1] if val >= 0 else PALETTE[5]
            ax.barh(i, val, left=cumulative if val >= 0 else cumulative + val,
                    color=color, edgecolor="black")
            cumulative += val
        ax.set_yticks(range(len(top))); ax.set_yticklabels(top.index, fontsize=8)
        ax.axvline(base_rate, color="gray", linestyle="--", label=f"Base rate ({base_rate:.3f})")
        ax.axvline(prob[idx], color="black", linestyle="-", label=f"Prediction ({prob[idx]:.3f})")
        ax.set_title(title, fontsize=10)
        ax.legend(fontsize=7)
    fig.suptitle("Local Waterfall Explanations (Integrated Gradients attributions)", fontweight="bold")
    fig.tight_layout()
    save_fig(fig, "fig21_local_waterfall")



def main():
    seed_everything(SEED)
    df, target_col = load_ibm_hr_dataset(CSV_PATH)

    (fold_metrics, last_fold_curves, last_fold_artifacts,
     feature_names, num_cols, cat_cols, cat_cardinalities, best_cfg) = run_experiment(df, target_col)

    mean_df, std_df = aggregate_metrics(fold_metrics)
    save_table(mean_df, "table01_mean_metrics")
    save_table(std_df, "table02_std_metrics")

    per_fold_rows = []
    for name, folds in fold_metrics.items():
        for i, m in enumerate(folds):
            row = {"Model": name, "Fold": i + 1}
            row.update(m)
            per_fold_rows.append(row)
    save_table(pd.DataFrame(per_fold_rows).set_index(["Model", "Fold"]), "table03_per_fold_metrics")

    ranking = mean_df.rank(ascending=False, method="min")
    save_table(ranking, "table04_metric_rankings")

    METRICS_LIST = ["Accuracy", "Precision", "Recall", "F1", "AUC_ROC", "AUC_PR", "MCC", "BalancedAcc"]

    save_table(combined_mean_std_table(mean_df, std_df), "table06_mean_std_combined")

    sig_f1 = pairwise_significance_table(fold_metrics, "F1", PROPOSED_NAME, BASELINE_NAMES)
    save_table(sig_f1.set_index("Comparison"), "table07_significance_F1")
    sig_auc = pairwise_significance_table(fold_metrics, "AUC_ROC", PROPOSED_NAME, BASELINE_NAMES)
    save_table(sig_auc.set_index("Comparison"), "table08_significance_AUC_ROC")

    rel_improve = relative_improvement_table(mean_df, PROPOSED_NAME, BASELINE_NAMES)
    save_table(rel_improve, "table09_relative_improvement")

    abl_compare = ablation_comparison_table(mean_df, PROPOSED_NAME, ABLATION_NAME)
    save_table(abl_compare, "table10_ablation_comparison")

    salience_table = extract_feature_salience(
        last_fold_artifacts["model"], last_fold_artifacts["xn"], last_fold_artifacts["xc"], feature_names
    )
    save_table(salience_table, "table11_feature_salience")

    bootstrap_df = bootstrap_ci_table(fold_metrics, METRICS_LIST, ALL_MODEL_NAMES)
    save_table(bootstrap_df, "table12_bootstrap_ci")

    cohend_df = cohend_effect_size_table(fold_metrics, METRICS_LIST, PROPOSED_NAME, BASELINE_NAMES)
    save_table(cohend_df, "table13_effect_size_cohend")

    ranks_f1, friedman_f1 = friedman_nemenyi_tables(fold_metrics, "F1", ALL_MODEL_NAMES)
    save_table(ranks_f1, "table14a_friedman_ranks_F1")
    save_table(friedman_f1, "table14b_friedman_summary_F1")
    ranks_auc, friedman_auc = friedman_nemenyi_tables(fold_metrics, "AUC_ROC", ALL_MODEL_NAMES)
    save_table(ranks_auc, "table14c_friedman_ranks_AUC_ROC")
    save_table(friedman_auc, "table14d_friedman_summary_AUC_ROC")

    n_num = len(num_cols)
    complexity_df = complexity_latency_table(n_num, cat_cardinalities, best_cfg)
    save_table(complexity_df, "table15_complexity_latency")

    pub_summary_df, _ = publication_summary_table(
        fold_metrics, mean_df, std_df, METRICS_LIST, PROPOSED_NAME, BASELINE_NAMES
    )
    save_table(pub_summary_df, "table16_publication_summary")

    print("\n===== MEAN METRICS (across {}x{} folds) =====".format(N_FOLDS, N_REPEATS))
    print(mean_df.round(4).to_string())

    proposed_f1 = mean_df.loc[PROPOSED_NAME, "F1"]
    if proposed_f1 >= 0.95:
        print("\n*** SANITY WARNING ***")
        print(f"{PROPOSED_NAME} reported mean F1={proposed_f1:.4f} on this dataset, which is far above "
              "anything published without leakage. Before trusting this number: re-check that the "
              "scaler/encoder were fit only on training folds, that thresholds were tuned only on "
              "validation splits, and that there are no duplicate rows split across train/test. "
              "Do not report this number until you've ruled those out.")
    else:
        print(f"\n{PROPOSED_NAME} mean F1 = {proposed_f1:.4f}. This is the expected, honest range for "
              "this dataset -- compare it to your original CA-FTX (0.5514) to see the real gain from "
              "the gated feature-interaction layer, PSO-tuned hyperparameters, and ensembling.")

    plot_roc_pr(last_fold_curves)
    plot_confusion(last_fold_curves)
    plot_cv_box(fold_metrics, metric="F1")
    plot_cv_box(fold_metrics, metric="AUC_ROC")
    plot_mean_f1_ranked(mean_df)
    pso_hist = pd.read_csv(os.path.join(TAB_DIR, "table05_pso_convergence.csv"))
    plot_pso_convergence(pso_hist)
    permutation_importance_proposed(last_fold_artifacts, feature_names, num_cols, cat_cols)

    last_model = last_fold_artifacts["model"]
    last_xn, last_xc = last_fold_artifacts["xn"], last_fold_artifacts["xc"]
    last_y, last_thr = last_fold_artifacts["y"], last_fold_artifacts["threshold"]
    last_prob = predict_prob(last_model, last_xn, last_xc)
    last_pred = (last_prob >= last_thr).astype(int)

    plot_feature_salience_bar(salience_table)                                  

    shap_attrs = shap_style_attributions(last_model, last_xn, last_xc)
    ig_attrs = integrated_gradients_attributions(last_model, last_xn, last_xc)
    plot_shap_global_bar(shap_attrs, feature_names)                            
    plot_ig_directional_bar(ig_attrs, feature_names)                           
    plot_calibration_curves(last_fold_curves)                                
    plot_dataset_overview(df, target_col, num_cols)                            

    forest_metrics = ["F1", "AUC_ROC", "AUC_PR", "Precision", "Recall", "MCC"]
    plot_forest(bootstrap_df, forest_metrics, PROPOSED_NAME, ABLATION_NAME, BASELINE_NAMES)  

    radar_metrics = ["Accuracy", "Precision", "Recall", "F1", "AUC_ROC", "AUC_PR", "BalancedAcc"]
    plot_radar(mean_df, radar_metrics, PROPOSED_NAME)                         

    plot_cd_diagram(ranks_f1, "F1")                                            
    plot_cd_diagram(ranks_auc, "AUC_ROC")                                      
    plot_spaghetti(fold_metrics, "F1", ALL_MODEL_NAMES)                        
    plot_spaghetti(fold_metrics, "AUC_ROC", ALL_MODEL_NAMES)                   

    ablation_metrics = ["F1", "AUC_ROC", "Precision", "Recall"]
    plot_ablation_slope(fold_metrics, ablation_metrics, PROPOSED_NAME, ABLATION_NAME)  

    plot_tsne_embedding(last_model, last_xn, last_xc, last_y, last_prob)       
    plot_beeswarm(shap_attrs, last_xn, last_xc, feature_names, num_cols)       
    plot_explanation_agreement(shap_attrs, ig_attrs, feature_names)           
    plot_local_waterfall(last_y, last_prob, last_pred, ig_attrs, feature_names,
                          base_rate=float(last_prob.mean()))                   

    print("\nBest PSO-found configuration for", PROPOSED_NAME, ":", json.dumps(best_cfg, indent=2))
    print("All figures saved under:", FIG_DIR)
    print("All tables saved under:", TAB_DIR)

    zip_path = "outputs.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for root, _, files_ in os.walk(OUT_DIR):
            for f in files_:
                full = os.path.join(root, f)
                zf.write(full, os.path.relpath(full, "."))
    print(f"\nDone. Zipped all outputs to: {zip_path}")


if __name__ == "__main__":
    main()

Using device: cuda
Loaded WA_Fn-UseC_-HR-Employee-Attrition.csv -- shape=(1470, 35), positive rate=0.161, exact duplicate rows=0

--- PSO metaheuristic search: 8 particles x 6 iterations (48 short trainings on an inner split) ---
  iter 1/6 particle 1/8: val F1=0.5918 arch=(32, 4, 2) lr=7.66e-04
  iter 1/6 particle 2/8: val F1=0.5250 arch=(64, 8, 3) lr=1.07e-04
  iter 1/6 particle 3/8: val F1=0.5714 arch=(24, 4, 1) lr=5.96e-04
  iter 1/6 particle 4/8: val F1=0.5205 arch=(16, 4, 1) lr=4.72e-04
  iter 1/6 particle 5/8: val F1=0.5870 arch=(48, 4, 2) lr=1.79e-04
  iter 1/6 particle 6/8: val F1=0.5783 arch=(64, 8, 2) lr=1.02e-03
  iter 1/6 particle 7/8: val F1=0.5897 arch=(16, 2, 1) lr=9.52e-04
  iter 1/6 particle 8/8: val F1=0.5682 arch=(24, 4, 1) lr=2.44e-03
  iter 2/6 particle 1/8: val F1=0.5918 arch=(32, 4, 2) lr=7.66e-04
  iter 2/6 particle 2/8: val F1=0.5833 arch=(32, 4, 1) lr=8.03e-04
  iter 2/6 particle 3/8: val F1=0.5714 arch=(32, 4, 1) lr=7.67e-04
  iter 2/6 particle 4/8: val F1=0